# cycle-detection-temp-set — faded example 2: Complete the un-gray step so siblings aren't false-flagged

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `cycle-detection-temp-set`. Running the beacon reports progress on the `Backprop: cycle detection via temp set` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: cycle detection via temp set` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cycle-detection-temp-set`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cycle-detection-temp-set"
DD_SUBTOPIC = "Backprop: cycle detection via temp set"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

When a DFS finishes a vertex's whole subtree, that vertex leaves the recursion stack and must be removed from `temp` and added to `perm`. If you skip removing it from `temp`, a sibling path that legitimately reaches the same vertex will wrongly see it as 'on the stack' and report a phantom cycle. The un-gray step is what makes diamond DAGs pass.

## Faded exercise 2

Implement `toposort(adj)` that returns a topological order and raises `ValueError` on a cycle. The gray check, the recursion, and the emit logic are provided. **Complete the un-gray transition** that runs as the DFS leaves a vertex — moving it off the stack and marking its subtree finished.

**Fill in:** Removes the just-finished vertex from temp (it is no longer on the stack) and records it in perm as fully processed.

In [ ]:
def toposort(adj):
    perm, temp, order = set(), set(), []

    def visit(u):
        if u in perm:
            return
        if u in temp:
            raise ValueError(f"cycle through {u}")
        temp.add(u)
        for v in adj.get(u, []):
            visit(v)
        temp.remove(u)
        perm.add(u)
        order.append(u)

    for u in adj:
        visit(u)
    order.reverse()
    return order


def _test():
    dag = {"a": ["b", "c"], "b": ["d"], "c": ["d"], "d": []}
    order = toposort(dag)
    pos = {n: i for i, n in enumerate(order)}
    assert set(order) == {"a", "b", "c", "d"}, "all nodes present once"
    assert len(order) == 4, "no duplicates -- un-gray must not break perm marking"
    for u, vs in dag.items():
        for v in vs:
            assert pos[u] < pos[v], f"edge {u}->{v} must point forward"
    cycle = False
    try:
        toposort({"p": ["q"], "q": ["p"]})
    except ValueError:
        cycle = True
    assert cycle, "cyclic graph must raise"


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def toposort(adj):
    perm, temp, order = set(), set(), []

    def visit(u):
        if u in perm:
            return
        if u in temp:
            raise ValueError(f"cycle through {u}")
        temp.add(u)
        for v in adj.get(u, []):
            visit(v)
        temp.remove(u)
        perm.add(u)
        order.append(u)

    for u in adj:
        visit(u)
    order.reverse()
    return order
```
</details>